# Data Owner 2 (DO2) - Few-shot Client

This notebook is pre-configured for **DO 2 (Few-shot)** participating in federated SAM2LoRA training.

## Client Configuration
- **Type**: Few-shot (memory bank, no gradient training)
- **Training samples**: 4
- **Test samples**: 2
- **Contribution**: Does NOT contribute weights to aggregation

## How Few-shot Works
- Builds a memory bank from labeled examples
- Uses similarity matching for segmentation
- No gradient-based training required

## Prerequisites
1. Go to https://colab.research.google.com/
2. Upload this notebook with `File` -> `Upload Notebook`

## Install Dependencies

In [ ]:
!uv pip install -v "git+https://github.com/OpenMined/syft-flwr.git@feat/syft-client-p2p" 2>&1 | grep -E "(OpenMined/syft-flwr|OpenMined/syft-client).*[0-9a-f]{7}"

## Login to Datasite

In [ ]:
import syft_client as sc
import syft_flwr

print(f"{sc.__version__ = }")
print(f"{syft_flwr.__version__ = }")

# do_email = input("Enter DO2's email: ")
do_email = "your.do2.fewshot@gmail.com"  # Replace with your email
do_client = sc.login_do(email=do_email)

## View Peers

In [ ]:
do_client.peers

## Dataset Configuration

**DO2 Few-shot Configuration:**
- 4 training samples (for memory bank)
- 2 test samples
- Uses memory bank for similarity-based segmentation

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

DATASET_DIR = Path("./dataset/").expanduser().absolute()

if not DATASET_DIR.exists():
    print("Downloading Chest CT Segmentation dataset...")
    snapshot_download(
        repo_id="khoaguin/chest-ct-segmentation",
        repo_type="dataset",
        local_dir=DATASET_DIR,
    )

DATASET_PATH = DATASET_DIR / "chest-ct-segmentation"
print(f"Dataset path: {DATASET_PATH}")

In [ ]:
# DO2 Configuration - Few-shot
CLIENT_TYPE = "DO_2_fewshot"
TRAIN_SAMPLES = 4
TEST_SAMPLES = 2

print(f"Client Type: {CLIENT_TYPE}")
print(f"Training samples: {TRAIN_SAMPLES}")
print(f"Test samples: {TEST_SAMPLES}")
print(f"Method: Memory bank (no gradient training)")
print(f"Contributes to aggregation: NO")

In [ ]:
# Upload dataset
do_client.create_dataset(
    name="chest-ct-segmentation",
    mock_path=DATASET_PATH / "mock" if (DATASET_PATH / "mock").exists() else DATASET_PATH,
    private_path=DATASET_PATH,
    summary=f"Chest CT Segmentation dataset for DO2 (Few-shot: {TRAIN_SAMPLES} samples)",
    readme_path=DATASET_PATH / "README.md" if (DATASET_PATH / "README.md").exists() else None,
    tags=["medical", "segmentation", "sam2", "ct", "fewshot"],
    sync=True,
)
print("Dataset uploaded!")

In [ ]:
do_client.datasets.get_all()

## Jobs

In [ ]:
do_client.jobs

In [ ]:
if len(do_client.jobs) > 0:
    print(do_client.jobs[0])

## Approve and Run Jobs

In [ ]:
if len(do_client.jobs) > 0:
    do_client.jobs[0].approve()
    print("Job approved!")

In [ ]:
do_client.process_approved_jobs()

In [ ]:
do_client.jobs

## Clean Up

In [ ]:
# do_client.delete_syftbox()